# Week 1 – Day 5-7: Python Data Pipeline
SQLAlchemy pipeline pulling PostgreSQL views into pandas with a data validation (contract) layer.
Outputs are cached to `pipeline_output/` for reproducibility downstream.

## 1. Configuration

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text, inspect
from pathlib import Path
import logging
import json
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(levelname)s – %(message)s')
logger = logging.getLogger(__name__)

# ── DB config ────────────────────────────────────────────────────────────────
DB_USER     = "olist_user"
DB_PASSWORD = "1234"
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "olist_db"

OUTPUT_DIR = Path("pipeline_output")
OUTPUT_DIR.mkdir(exist_ok=True)

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True,
    echo=False
)

# verify connection
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
logger.info("Connected to %s", DB_NAME)

INFO – Connected to olist_db


## 2. Data Contracts
Each view has an explicit contract: required columns, expected dtypes, and value constraints.
Contracts are the single source of truth — downstream code trusts these guarantees.

In [2]:
# dtype categories: 'numeric', 'datetime', 'string'
# constraints: {'not_null': [cols], 'positive': [cols], 'range': {col: (min, max)}}

CONTRACTS = {
    "monthly_revenue": {
        "required_columns": ["month", "revenue", "prev_month_revenue", "mom_growth_pct"],
        "dtypes": {
            "month":               "datetime",
            "revenue":             "numeric",
            "prev_month_revenue":  "numeric",
            "mom_growth_pct":      "numeric",
        },
        "constraints": {
            "not_null":  ["month", "revenue"],
            "positive":  ["revenue"],
        },
        "min_rows": 1,
    },
    "cohort_retention": {
        "required_columns": ["cohort_month", "months_since_first", "cohort_size", "retained"],
        "dtypes": {
            "cohort_month":        "datetime",
            "months_since_first":  "numeric",
            "cohort_size":         "numeric",
            "retained":            "numeric",
        },
        "constraints": {
            "not_null":  ["cohort_month", "cohort_size", "retained"],
            "positive":  ["cohort_size", "retained"],
            "range":     {"months_since_first": (0, 36)},
        },
        "min_rows": 1,
    },
    "seller_ranking": {
        "required_columns": ["seller_id", "total_orders", "total_revenue", "avg_review_score", "revenue_rank"],
        "dtypes": {
            "seller_id":       "string",
            "total_orders":    "numeric",
            "total_revenue":   "numeric",
            "avg_review_score": "numeric",
            "revenue_rank":    "numeric",
        },
        "constraints": {
            "not_null":  ["seller_id", "total_orders", "total_revenue", "revenue_rank"],
            "positive":  ["total_orders", "total_revenue", "revenue_rank"],
            "range":     {"avg_review_score": (1.0, 5.0)},
        },
        "min_rows": 1,
    },
    "delivery_delay_by_state": {
        "required_columns": ["customer_state", "total_orders", "avg_delay_days"],
        "dtypes": {
            "customer_state": "string",
            "total_orders":   "numeric",
            "avg_delay_days": "numeric",
        },
        "constraints": {
            "not_null":  ["customer_state", "total_orders"],
            "positive":  ["total_orders"],
        },
        "min_rows": 1,
    },
}

print(f"Contracts defined for {len(CONTRACTS)} views.")

Contracts defined for 4 views.


## 3. Validator

In [3]:
class ValidationError(Exception):
    pass


def validate(df: pd.DataFrame, view_name: str, contract: dict) -> list[str]:
    """Run all contract checks. Returns list of warning strings (empty = pass)."""
    warnings = []

    # ── row count ────────────────────────────────────────────────────────────
    min_rows = contract.get("min_rows", 1)
    if len(df) < min_rows:
        raise ValidationError(f"{view_name}: expected >= {min_rows} rows, got {len(df)}")

    # ── required columns ─────────────────────────────────────────────────────
    missing_cols = [c for c in contract["required_columns"] if c not in df.columns]
    if missing_cols:
        raise ValidationError(f"{view_name}: missing columns {missing_cols}")

    # ── dtype checks ─────────────────────────────────────────────────────────
    for col, expected_kind in contract.get("dtypes", {}).items():
        if col not in df.columns:
            continue
        actual = df[col].dtype
        ok = (
            (expected_kind == "numeric"  and pd.api.types.is_numeric_dtype(actual)) or
            (expected_kind == "datetime" and pd.api.types.is_datetime64_any_dtype(actual)) or
            (expected_kind == "string"   and (pd.api.types.is_string_dtype(actual) or pd.api.types.is_object_dtype(actual)))
        )
        if not ok:
            warnings.append(f"{view_name}.{col}: expected {expected_kind}, got {actual}")

    constraints = contract.get("constraints", {})

    # ── null checks ──────────────────────────────────────────────────────────
    for col in constraints.get("not_null", []):
        if col not in df.columns:
            continue
        n_null = df[col].isna().sum()
        if n_null > 0:
            warnings.append(f"{view_name}.{col}: {n_null} null values")

    # ── positivity checks ────────────────────────────────────────────────────
    for col in constraints.get("positive", []):
        if col not in df.columns:
            continue
        n_bad = (df[col].dropna() <= 0).sum()
        if n_bad > 0:
            warnings.append(f"{view_name}.{col}: {n_bad} non-positive values")

    # ── range checks ─────────────────────────────────────────────────────────
    for col, (lo, hi) in constraints.get("range", {}).items():
        if col not in df.columns:
            continue
        series = df[col].dropna()
        n_out = ((series < lo) | (series > hi)).sum()
        if n_out > 0:
            warnings.append(f"{view_name}.{col}: {n_out} values outside [{lo}, {hi}]")

    return warnings


print("Validator ready.")

Validator ready.


## 4. Pipeline: Pull Views → Validate → Cache

In [4]:
def pull_view(view_name: str, engine) -> pd.DataFrame:
    with engine.connect() as conn:
        df = pd.read_sql(text(f"SELECT * FROM {view_name}"), conn)
    logger.info("Pulled %-28s  %d rows × %d cols", view_name, *df.shape)
    return df


def run_pipeline(views: list[str], contracts: dict, engine, output_dir: Path) -> dict:
    """Pull each view, validate, cache to CSV. Returns {view: df} on success."""
    results = {}
    run_meta = {"timestamp": datetime.utcnow().isoformat(), "views": {}}

    for view in views:
        logger.info("─── Processing: %s", view)

        # pull
        df = pull_view(view, engine)

        # validate
        contract = contracts[view]
        warnings = validate(df, view, contract)

        if warnings:
            for w in warnings:
                logger.warning(w)
        else:
            logger.info("%s: all checks passed", view)

        # cache
        out_path = output_dir / f"{view}.csv"
        df.to_csv(out_path, index=False)
        logger.info("Cached → %s", out_path)

        results[view] = df
        run_meta["views"][view] = {
            "rows":     len(df),
            "cols":     list(df.columns),
            "warnings": warnings,
        }

    # write run manifest
    manifest_path = output_dir / "run_manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(run_meta, f, indent=2)
    logger.info("Manifest written → %s", manifest_path)

    return results


VIEWS = list(CONTRACTS.keys())
dataframes = run_pipeline(VIEWS, CONTRACTS, engine, OUTPUT_DIR)

INFO – ─── Processing: monthly_revenue
INFO – Pulled monthly_revenue               22 rows × 4 cols
INFO – monthly_revenue: all checks passed
INFO – Cached → pipeline_output\monthly_revenue.csv
INFO – ─── Processing: cohort_retention
INFO – Pulled cohort_retention              225 rows × 4 cols
INFO – cohort_retention: all checks passed
INFO – Cached → pipeline_output\cohort_retention.csv
INFO – ─── Processing: seller_ranking
INFO – Pulled seller_ranking                3090 rows × 5 cols
INFO – seller_ranking: all checks passed
INFO – Cached → pipeline_output\seller_ranking.csv
INFO – ─── Processing: delivery_delay_by_state
INFO – Pulled delivery_delay_by_state       27 rows × 3 cols
INFO – delivery_delay_by_state: all checks passed
INFO – Cached → pipeline_output\delivery_delay_by_state.csv
INFO – Manifest written → pipeline_output\run_manifest.json


## 5. Quick Inspection

In [5]:
# ── monthly_revenue ───────────────────────────────────────────────────────────
df_rev = dataframes["monthly_revenue"]
print("monthly_revenue — shape:", df_rev.shape)
df_rev.head()

monthly_revenue — shape: (22, 4)


,month,revenue,prev_month_revenue,mom_growth_pct
0,2016-10-01,46566.71,NaN,NaN
1,2016-12-01,19.62,46566.71,-99.96
2,2017-01-01,127545.67,19.62,649979.87
3,2017-02-01,271298.65,127545.67,112.71
4,2017-03-01,414369.39,271298.65,52.74


In [6]:
# ── cohort_retention ──────────────────────────────────────────────────────────
df_cohort = dataframes["cohort_retention"]
print("cohort_retention — shape:", df_cohort.shape)
df_cohort.head()

cohort_retention — shape: (225, 4)


,cohort_month,months_since_first,cohort_size,retained
0,2016-09-01,0.000000,4,4
1,2016-10-01,0.000000,321,321
2,2016-10-01,6.066667,1,1
3,2016-10-01,9.100000,1,1
4,2016-10-01,11.166667,1,1


In [7]:
# ── seller_ranking ────────────────────────────────────────────────────────────
df_sellers = dataframes["seller_ranking"]
print("seller_ranking — shape:", df_sellers.shape)
df_sellers.head(10)

seller_ranking — shape: (3090, 5)


,seller_id,total_orders,total_revenue,avg_review_score,revenue_rank
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,228071.04,4.12,1
1,53243585a1d6dc2643021fd1853d8905,356,220740.05,4.08,2
2,4a3ca9315b744ce9f8e9374361493884,1785,200561.42,3.80,3
3,fa1c13f2614d7b5c4749cbc52fecda94,581,192774.43,4.34,4
4,7c67e1448b00f6e969d365cea6b010ab,976,188017.85,3.35,5
5,7e93a43ef30c4f03f38b393420bc753a,335,176201.88,4.21,6
6,da8622b14eb17ae2831f4ac5b9dab84a,1308,161993.97,4.07,7
7,7a67c85e85bb2ce8582c35f2203ad736,1151,141130.58,4.23,8
8,1025f0e2d44d7041d6cf58b6550e0bfa,907,139484.38,3.85,9
9,955fee9216a65b617aa5c0531780ce60,1277,133948.81,4.05,10


In [8]:
# ── delivery_delay_by_state ───────────────────────────────────────────────────
df_delay = dataframes["delivery_delay_by_state"]
print("delivery_delay_by_state — shape:", df_delay.shape)
df_delay.head(10)

delivery_delay_by_state — shape: (27, 3)


,customer_state,total_orders,avg_delay_days
0,AL,397,-8.03
1,MA,717,-8.89
2,SE,335,-9.33
3,ES,1995,-9.80
4,BA,3256,-10.10
5,CE,1279,-10.11
6,MS,701,-10.36
7,SP,40495,-10.38
8,PI,476,-10.63
9,SC,3547,-10.81


## 6. Pipeline Summary

In [9]:
summary_rows = []
for view, df in dataframes.items():
    warnings = validate(df, view, CONTRACTS[view])  # re-run for display
    null_counts = df.isna().sum()
    summary_rows.append({
        "view":          view,
        "rows":          len(df),
        "cols":          df.shape[1],
        "total_nulls":   int(null_counts.sum()),
        "null_cols":     [c for c, n in null_counts.items() if n > 0],
        "warnings":      len(warnings),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n=== Pipeline Summary ===")
summary_df


=== Pipeline Summary ===


,view,rows,cols,total_nulls,null_cols,warnings
0,monthly_revenue,22,4,2,"[prev_month_revenue, mom_growth_pct]",0
1,cohort_retention,225,4,0,[],0
2,seller_ranking,3090,5,0,[],0
3,delivery_delay_by_state,27,3,0,[],0


In [10]:
# ── revenue stats ────────────────────────────────────────────────────────────
print("Revenue stats:")
df_rev[["revenue", "mom_growth_pct"]].describe().round(2)

Revenue stats:


,revenue,mom_growth_pct
count,22.00,21.00
mean,701020.99,30960.21
std,375804.94,141835.23
min,19.62,-99.96
25%,433333.44,-5.65
50%,726155.13,7.13
75%,1023950.56,27.92
max,1153528.05,649979.87


In [11]:
# ── cohort: retention rate column ────────────────────────────────────────────
df_cohort["retention_rate"] = (df_cohort["retained"] / df_cohort["cohort_size"]).round(4)
print("Retention rate range:", df_cohort["retention_rate"].min(), "–", df_cohort["retention_rate"].max())
df_cohort[["cohort_month", "months_since_first", "cohort_size", "retained", "retention_rate"]].head(12)

Retention rate range: 1.0 – 1.0


,cohort_month,months_since_first,cohort_size,retained,retention_rate
0,2016-09-01,0.000000,4,4,1.0
1,2016-10-01,0.000000,321,321,1.0
2,2016-10-01,6.066667,1,1,1.0
3,2016-10-01,9.100000,1,1,1.0
4,2016-10-01,11.166667,1,1,1.0
5,2016-10-01,13.200000,1,1,1.0
6,2016-10-01,15.233333,1,1,1.0
7,2016-10-01,17.200000,1,1,1.0
8,2016-10-01,19.233333,2,2,1.0
9,2016-10-01,20.266667,2,2,1.0


In [12]:
# re-cache cohort with retention_rate added
df_cohort.to_csv(OUTPUT_DIR / "cohort_retention.csv", index=False)
print("cohort_retention.csv updated with retention_rate column.")

cohort_retention.csv updated with retention_rate column.


In [13]:
# ── delivery: top 5 states by delay ─────────────────────────────────────────
print("Top 5 states — avg delivery delay:")
print(df_delay.nlargest(5, "avg_delay_days")[["customer_state", "avg_delay_days", "total_orders"]].to_string(index=False))
print("\nTop 5 states — fewest delay:")
print(df_delay.nsmallest(5, "avg_delay_days")[["customer_state", "avg_delay_days", "total_orders"]].to_string(index=False))

Top 5 states — avg delivery delay:
customer_state  avg_delay_days  total_orders
            AL           -8.03           397
            MA           -8.89           717
            SE           -9.33           335
            ES           -9.80          1995
            BA          -10.10          3256

Top 5 states — fewest delay:
customer_state  avg_delay_days  total_orders
            AC          -20.08            80
            RO          -19.40           243
            AP          -19.06            67
            AM          -18.85           145
            RR          -16.59            41


In [14]:
print("\nPipeline complete.")
print(f"Outputs in: {OUTPUT_DIR.resolve()}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")


Pipeline complete.
Outputs in: D:\Programming\Projects\E-commerce_project\pipeline_output
  cohort_retention.csv
  delivery_delay_by_state.csv
  monthly_revenue.csv
  run_manifest.json
  seller_ranking.csv
